In [2]:
#Step 1 — Query candidates from database
#Connects to the CHAMPSS database using CandidateViewerQuery, loops over a date range (folders), and collects all candidates matching #specific classifications (<faint>, NEW CANDIDATE) into a single list (all_candidates).
#Step 2 — Run multi-day folding pipeline
#For each candidate, builds arguments and runs multidayfold_pipeline to process data across multiple days; skips candidates already #processed (based on existing output files) and stores results in all_outputs.
#Step 3 — Save results
#Serializes the collected outputs (all_outputs) into a file using pickle, allowing the results to be reloaded later without recomputing.

import sps_databases
import subprocess
from cfbm.bm_data import get_data
import os
import numpy as np
from sps_databases import db_utils, db_api
import scipy
from datetime import datetime, timedelta
from scheduler.run_as_service import run_as_service
import pickle

In [2]:
# Before running any scripts that call schedule_workflow_job outside of a container, you'll need to run
!workflow workspace set champss.workspace.yml #puting !run the command in bash

Currently using champss workspace.
Locating workspace champss.workspace.yml
Workspace champss.workspace.yml not found.


In [3]:
#All harcoded info should be in the CONFIG dict, so that it is easier to change if needed and to avoid hardcoding in the code itself.
config = {
    "db_config": {
        "host": "sps-archiver1",
        "user": "automation",
        "port": 3306,
        "password": "",
        "database": "champss",
    },
    "start_date": datetime(2026, 4, 9),#3,22
    "end_date": datetime(2026, 6, 15),#4,8
    "classifications": ["<faint>", "NEW CANDIDATE"],
    "outfile_dir": "/mnt/beegfs-client/processed/multiday/",
    "raw_dir": "/mnt/beegfs-client/raw/",
    "archive_dir": "/mnt/beegfs-client/processed/archives/",
    "docker_image": "sps-archiver1.chime:5000/champss_software:run_on_compute1",
}


#Step_1-Query website candidates and put them in a list
from sps_pipeline.candidate_viewer import CandidateViewerQuery
from sps_pipeline.candidate_viewer import CandidateViewerRegistrar
from multiday_search import multidayfold_pipeline


# Database configuration
db_config = {
    'host': 'sps-archiver1',
    'user': 'automation',
    'port': 3306,
    'password': '',#no password for automation user
    'database': 'champss'
}

#Creating a list of date
start_date = config["start_date"]
end_date   = config["end_date"]

folders = []
current = start_date

while current <= end_date:
    folders.append(current.strftime("%Y-%m-%d"))
    current += timedelta(days=1)
    
#query = CandidateViewerQuery(survey="stackcands", db_config=db_config)
#candidates = query.get_metadata(folder="stack_0")
classifications = config["classifications"]

all_candidates = []
with CandidateViewerQuery(survey='dailycands', db_config=db_config) as query:
    for folder in folders:
        print(f"\nProcessing folder: {folder}")

        for cls in classifications:
            try:
                candidates = query.get_ratings(
                    folder=folder,
                    classification=cls,
                    with_metadata=True
                )
            except Exception:
                print(f"No data for {folder}")
                continue  # skip this classification if query fails

            
            print(f"Found {len(candidates)} candidates for {cls} in {folder}")
            all_candidates.extend(candidates)

print("Query finished")

16 Jun 2026 14:03:38 UTC INFO      mysql.connector package: mysql.connector.plugins

package: mysql.connector.plugins
package: mysql.connector.plugins
package: mysql.connector.plugins


16 Jun 2026 14:03:38 UTC INFO      mysql.connector plugin_name: mysql_native_password

plugin_name: mysql_native_password
plugin_name: mysql_native_password
plugin_name: mysql_native_password


16 Jun 2026 14:03:38 UTC INFO      mysql.connector AUTHENTICATION_PLUGIN_CLASS: MySQLNativePasswordAuthPlugin

AUTHENTICATION_PLUGIN_CLASS: MySQLNativePasswordAuthPlugin
AUTHENTICATION_PLUGIN_CLASS: MySQLNativePasswordAuthPlugin
AUTHENTICATION_PLUGIN_CLASS: MySQLNativePasswordAuthPlugin



Processing folder: 2026-04-09
Found 0 candidates for <faint> in 2026-04-09
Found 0 candidates for NEW CANDIDATE in 2026-04-09

Processing folder: 2026-04-10
Found 0 candidates for <faint> in 2026-04-10
Found 0 candidates for NEW CANDIDATE in 2026-04-10

Processing folder: 2026-04-11
Found 0 candidates for <faint> in 2026-04-11
Found 0 candidates for NEW CANDIDATE in 2026-04-11

Processing folder: 2026-04-12
Found 0 candidates for <faint> in 2026-04-12
Found 0 candidates for NEW CANDIDATE in 2026-04-12

Processing folder: 2026-04-13
Found 0 candidates for <faint> in 2026-04-13
Found 0 candidates for NEW CANDIDATE in 2026-04-13

Processing folder: 2026-04-14
Found 0 candidates for <faint> in 2026-04-14
Found 0 candidates for NEW CANDIDATE in 2026-04-14

Processing folder: 2026-04-15
Found 0 candidates for <faint> in 2026-04-15
Found 0 candidates for NEW CANDIDATE in 2026-04-15

Processing folder: 2026-04-16
Found 0 candidates for <faint> in 2026-04-16
Found 0 candidates for NEW CANDIDAT

In [4]:
for cand in all_candidates:
    print(cand['metadata']

{'survey': 'dailycands', 'folder': '2026-03-22', 'file': 'Multi_Pointing_Groups_f_11.707_DM_62.540_69c11f03ff2c065afeea3388', 'input_file': '/mnt/beegfs-client/processed/mp_runs/daily_20260322/candidates/Multi_Pointing_Groups_f_11.707_DM_62.540_69c11f03ff2c065afeea3388.npz', 'candidate': '', 'telescope': 'chime', 'epoch_topo': '', 'epoch_bary': '', 't_sample': '', 'data_folded': '', 'data_avg': '', 'data_stdev': '', 'profile_bins': '', 'profile_avg': '', 'profile_stdev': '', 'reduce_chi_sqr': '', 'prob_noise': '8.009849548339844', 'best_dm': '62.54032266021014', 'p_topo': '', 'p_topo_d1': '', 'p_topo_d2': '', 'p_bary': '85.4224041634209', 'p_bary_d1': '0', 'p_bary_d2': '0', 'p_orb': '', 'asin': '', 'eccentricity': '', 'w': '', 't_peri': '', 'header_size': '', 'data_size': '', 'data_type': '', 'notes': {'fs_id': '69c19570ff2c065afef822f4', 'fs_sigma': 8.009849548339844, 'fs_file': '/mnt/beegfs-client/processed/archives//candidates/203.74_56.42/cand_11.71_62.54_2026-03-22.ar'}, 'datataki

In [ ]:
#Step_2-Run the multi-day fold
all_outputs = []
for cand in all_candidates:
    metadata = cand['metadata']
    input_file = metadata['input_file']
    outfile = f"/mnt/beegfs-client/processed/multiday/{metadata['file']}"
    print(input_file)

    # Skip if already folded
    if os.path.exists(outfile):
        print(f"Skipping {metadata['file']} — already folded")
        continue

    print(f"\nRunning multidayfold_pipeline for {metadata['file']}...")

    #to run in terminal(need conversion to python file also:)command = f"multidayfold_pipeline --candpath {input_file}
    #--db-name champss_processing --nday 0 --datpath /mnt/beegfs-client/raw/ --foldpath /mnt/beegfs-client/processed/archives/ --use-workflow"
    args = [
    "--candpath",
    input_file,
    "--db-name",
    "champss_processing",
    "--nday",
    "0",   
    "--datpath",
    "/mnt/beegfs-client/raw/",
    "--foldpath",
    "/mnt/beegfs-client/processed/archives/",
    "--use-workflow",
    "--docker-image-name",
    "sps-archiver1.chime:5000/champss_software:run_on_compute1"
]

    #nday=0 is to run over all available days
    #you are starting a service that starts another service. 
    #The first one will be on sps-compute1 if you start it from your branch, but the second not
    #so we put the docker command so that both are on compute1

    #We want/need to add something to avoid rerunning the fold on candidate we did on previous day (the flag!!!)
    # Running the command
    try:
        fold_output = multidayfold_pipeline.main(
        args=args,
        standalone_mode=False
    )
        if fold_output is None:
            print(f"[ERROR] fold_output is None for {metadata['file']}")
            continue

        all_outputs.append([cand, fold_output[0]])
        
        print(f"Fold finished. Output should be at: {outfile}")
    except Exception as e:
        print(f"Folding failed for: {metadata['file']}")
        print(e)
        #add workflow option to the command to avoid error

/mnt/beegfs-client/processed/mp_runs/daily_20260422/candidates/Multi_Pointing_Groups_f_2.614_DM_243.077_69f45f946109e1d023b863f8.npz

Running multidayfold_pipeline for Multi_Pointing_Groups_f_2.614_DM_243.077_69f45f946109e1d023b863f8...
Source md_40.98_60.55_2.614494_243.08 already in the follow-up source database.


/home/rtellier/.cache/pypoetry/virtualenvs/champss-rqLHr3wD-py3.11/lib/python3.11/site-packages/pydantic/main.py:250: FutureWarning: WORKFLOW_TOKEN missing. Token auth will be required in the future.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


15 Jun 2026 18:06:37 UTC INFO      root Initial buckets entries: []

Initial buckets entries: []
Initial buckets entries: []
Initial buckets entries: []


15 Jun 2026 18:06:37 UTC INFO      root Final buckets entries: []

...

Creating Docker Service: 
{'image': 'sps-archiver1.chime:5000/champss_software:run_on_compute1', 'name': 'processing-fold-multiday-20260526-6a301a7c5541199611c9b838', 'command': 'workflow run champss-fold-multiday --tag 6a30563f82229b750bcd023e --site chime --lives 1 --sleep 1', 'env': ['CONTAINER_NAME={{.Task.Name}}', 'NODE_NAME={{.Node.Hostname}}'], 'mode': {'replicated': {'Replicas': 1}}, 'restart_policy': {'Condition': 'none', 'Delay': 0, 'MaxAttempts': 0, 'Window': 0}, 'labels': {'type': 'processing'}, 'constraints': ['node.hostname == sps-compute1'], 'resources': {'Reservations': {'MemoryBytes': 8000000000}}, 'mounts': [{'Target': '/var/run/docker.sock', 'Source': '/var/run/docker.sock', 'Type': 'bind', 'ReadOnly': False}, {'Target': '/dev/shm', 'Source': '', 'Type': 'tmpfs', 'ReadOnly': False, 'TmpfsOptions': {'SizeBytes': 100000000000}}, {'Target': '/mnt/beegfs-client/raw/', 'Source': '/mnt/beegfs-client/raw/', 'Type': 'bind', 'ReadOnly': False}, {'Target': '/mnt/beegfs-client/

15 Jun 2026 19:45:17 UTC INFO      root Creating Docker Service:                                                   
                                  {'image': 'sps-archiver1.chime:5000/champss_software:run_on_compute1', 'name':   
                                  'processing-fold-multiday-20260527-6a301a7c5541199611c9b838', 'command':         
                                  'workflow run champss-fold-multiday --tag 6a30564d82229b750bcd023f --site chime  
                                  --lives 1 --sleep 1', 'env': ['CONTAINER_NAME={{.Task.Name}}',                   
                                  'NODE_NAME={{.Node.Hostname}}'], 'mode': {'replicated': {'Replicas': 1}},        
                                  'restart_policy': {'Condition': 'none', 'Delay': 0, 'MaxAttempts': 0, 'Window':  
                                  0}, 'labels': {'type': 'processing'}, 'constraints': ['node.hostname ==          
                                  sps-compute1'], 'resources': {'Reservations': {'MemoryBytes': 8000000000}},      
                                  'mounts': [{'Target': '/var/run/docker.sock', 'Source': '/var/run/docker.sock',  
                                  'Type': 'bind', 'ReadOnly': False}, {'Target': '/dev/shm', 'Source': '', 'Type': 
                                  'tmpfs', 'ReadOnly': False, 'TmpfsOptions': {'SizeBytes': 100000000000}},        
                                  {'Target': '/mnt/beegfs-client/raw/', 'Source': '/mnt/beegfs-client/raw/',       
                                  'Type': 'bind', 'ReadOnly': False}, {'Target':                                   
                                  '/mnt/beegfs-client/processed/archives/', 'Source':                              
                                  '/mnt/beegfs-client/processed/archives/', 'Type': 'bind', 'ReadOnly': False}],   
                                  'networks': ['pipeline-network']}

Creating Docker Service: 
{'image': 'sps-archiver1.chime:5000/champss_software:run_on_compute1', 'name': 'processing-fold-multiday-20260527-6a301a7c5541199611c9b838', 'command': 'workflow run champss-fold-multiday --tag 6a30564d82229b750bcd023f --site chime --lives 1 --sleep 1', 'env': ['CONTAINER_NAME={{.Task.Name}}', 'NODE_NAME={{.Node.Hostname}}'], 'mode': {'replicated': {'Replicas': 1}}, 'restart_policy': {'Condition': 'none', 'Delay': 0, 'MaxAttempts': 0, 'Window': 0}, 'labels': {'type': 'processing'}, 'constraints': ['node.hostname == sps-compute1'], 'resources': {'Reservations': {'MemoryBytes': 8000000000}}, 'mounts': [{'Target': '/var/run/docker.sock', 'Source': '/var/run/docker.sock', 'Type': 'bind', 'ReadOnly': False}, {'Target': '/dev/shm', 'Source': '', 'Type': 'tmpfs', 'ReadOnly': False, 'TmpfsOptions': {'SizeBytes': 100000000000}}, {'Target': '/mnt/beegfs-client/raw/', 'Source': '/mnt/beegfs-client/raw/', 'Type': 'bind', 'ReadOnly': False}, {'Target': '/mnt/beegfs-client/

15 Jun 2026 19:45:39 UTC INFO      root Creating Docker Service:                                                   
                                  {'image': 'sps-archiver1.chime:5000/champss_software:run_on_compute1', 'name':   
                                  'processing-fold-multiday-20260528-6a301a7c5541199611c9b838', 'command':         
                                  'workflow run champss-fold-multiday --tag 6a30566382229b750bcd0240 --site chime  
                                  --lives 1 --sleep 1', 'env': ['CONTAINER_NAME={{.Task.Name}}',                   
                                  'NODE_NAME={{.Node.Hostname}}'], 'mode': {'replicated': {'Replicas': 1}},        
                                  'restart_policy': {'Condition': 'none', 'Delay': 0, 'MaxAttempts': 0, 'Window':  
                                  0}, 'labels': {'type': 'processing'}, 'constraints': ['node.hostname ==          
                                  sps-compute1'], 'resources': {'Reservations': {'MemoryBytes': 8000000000}},      
                                  'mounts': [{'Target': '/var/run/docker.sock', 'Source': '/var/run/docker.sock',  
                                  'Type': 'bind', 'ReadOnly': False}, {'Target': '/dev/shm', 'Source': '', 'Type': 
                                  'tmpfs', 'ReadOnly': False, 'TmpfsOptions': {'SizeBytes': 100000000000}},        
                                  {'Target': '/mnt/beegfs-client/raw/', 'Source': '/mnt/beegfs-client/raw/',       
                                  'Type': 'bind', 'ReadOnly': False}, {'Target':                                   
                                  '/mnt/beegfs-client/processed/archives/', 'Source':                              
                                  '/mnt/beegfs-client/processed/archives/', 'Type': 'bind', 'ReadOnly': False}],   
                                  'networks': ['pipeline-network']}

Creating Docker Service: 
{'image': 'sps-archiver1.chime:5000/champss_software:run_on_compute1', 'name': 'processing-fold-multiday-20260528-6a301a7c5541199611c9b838', 'command': 'workflow run champss-fold-multiday --tag 6a30566382229b750bcd0240 --site chime --lives 1 --sleep 1', 'env': ['CONTAINER_NAME={{.Task.Name}}', 'NODE_NAME={{.Node.Hostname}}'], 'mode': {'replicated': {'Replicas': 1}}, 'restart_policy': {'Condition': 'none', 'Delay': 0, 'MaxAttempts': 0, 'Window': 0}, 'labels': {'type': 'processing'}, 'constraints': ['node.hostname == sps-compute1'], 'resources': {'Reservations': {'MemoryBytes': 8000000000}}, 'mounts': [{'Target': '/var/run/docker.sock', 'Source': '/var/run/docker.sock', 'Type': 'bind', 'ReadOnly': False}, {'Target': '/dev/shm', 'Source': '', 'Type': 'tmpfs', 'ReadOnly': False, 'TmpfsOptions': {'SizeBytes': 100000000000}}, {'Target': '/mnt/beegfs-client/raw/', 'Source': '/mnt/beegfs-client/raw/', 'Type': 'bind', 'ReadOnly': False}, {'Target': '/mnt/beegfs-client/

In [ ]:
#Step_3-Retrieve the relevant parameters from the multiday fold

In [28]:
#Storing the output in a binary format instead of only in this file(which would be erased if disconnected)
with open('result_single_day5.pkl', 'wb') as outp:
    pickle.dump(all_outputs, outp, pickle.HIGHEST_PROTOCOL)

In [11]:
#Storing list of candidates in this format
#try:
#    with open('result_03.1.pkl', 'wb') as outp:
#        pickle.dump(all_candidates, outp, pickle.HIGHEST_PROTOCOL)
#except:
#    pass
#try:
#    np.save("result_03.1.npy", all_candidates, allow_pickle=True)
#except:
#    pass